<a href="https://colab.research.google.com/github/javism/MLexamples/blob/main/hopfield/ex08_Hopfield.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hopfield Networks

Notebook created by Jeff Orchand (University of Waterloo). See Jeff's lectures: 
* [Hopfield Networks (Theory)](https://www.youtube.com/watch?v=81B-ESqgCjs)
* [Hopfield Networks (demostration)](https://www.youtube.com/watch?v=0yklyZeKU1w)

## Preamble

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy

In [ ]:
# Useful functions
def IsScalar(x):
    if type(x) in (list, np.ndarray,):
        return False
    else:
        return True

def Thresh(x):
    if IsScalar(x):
        val = 1 if x>0 else -1
    else:
        val = np.ones_like(x)
        val[x<0] = -1.
    return val

def Hamming(x, y):
    '''
        d = Hamming(x,y)
        
        Hamming distance between two binary vectors x and y.
        It's the number of digits that differ.
        
        Inputs:
          x and y are arrays of binary vectors, and can be either {0,1} or {-1,1}
        
        Output:
          d is the number of places where the inputs differ
    '''
    d = []
    for xx, yy in zip(x,y):
        dd = 0.
        for xxx,yyy in zip(xx,yy):
            if xxx==1 and yyy!=1:
                dd += 1.
            elif yyy==1 and xxx!=1:
                dd += 1.
        d.append(dd)
    return d

def Perturb(x, p=0.1):
    '''
        y = Perturb(x, p=0.1)
        
        Apply binary noise to x. With probability p, each bit will be randomly
        set to -1 or 1.
        
        Inputs:
          x is an array of binary vectors of {-1,1}
          p is the probability of each bit being randomly flipped
        
        Output:
          y is an array of binary vectors of {-1,1}
    '''
    y = copy.deepcopy(x)
    for yy in y:
        for k in range(len(yy)):
            if np.random.rand()<p:
                yy[k] = Thresh(np.random.randint(2)*2-1)
    return y

# Toy Dataset

## Dataset options (choose one)

In [ ]:
# 4 Orthogonal Vectors
X = np.array([[ 1,  1,  1,  1,  1,  1,  1,  1, -1, -1, -1, -1, -1, -1, -1, -1],
              [ 1,  1,  1,  1, -1, -1, -1, -1,  1,  1,  1,  1, -1, -1, -1, -1],
              [ 1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1,  1, -1],
              [ 1,  1, -1, -1,  1,  1, -1, -1,  1,  1, -1, -1,  1,  1, -1, -1],
             ])
N = np.shape(X)[1]
n = 4

In [ ]:
X.shape

<div class="alert alert-block alert-info">
    After running the Hopfield network on the dataset above, try creating a random dataset, like the commented code below.
</div>

In [ ]:
# Random
# N = 50
# n = 6
# X = Thresh(np.random.normal(size=(n,N)))

## Set up network

In [ ]:
# (Pseudo-)Orthogonality
print(X@X.T)

In [ ]:
# Connection weights and biases
b = np.zeros((1,N))
b = np.sum(X, axis=0) / n
W = ( X.T @ X ) / n - np.eye(N)
W0 = copy.deepcopy(W)

In [ ]:
# This is equivalent to the above.
W1 = np.zeros_like(W)
for x in X:
    W1 += np.outer(x, x)/n
W1 -= np.identity(N)

print(np.max(W.flatten()-W1.flatten()))

In [ ]:
plt.subplot(121); plt.imshow(W, cmap='gray'); plt.axis('off');

In [ ]:
# Show that W@X returns a multiple of X
print(X@W)
print(np.array(X, dtype=float))

In [ ]:
def Energy(W, b, X):
    E = []
    for xx in X:
        blah = - 0.5 * (xx @ W)@xx.T + b@xx.T
        E.append(blah)
    return E

In [ ]:
def Update(W, x, b):
    xnew = x @ W - b
    return Thresh(xnew)

## Initial Network State (choose one)

In [ ]:
# Perturbed memory
k = np.random.randint(len(X))
Y = Perturb( X , p=0.4 )
x = Y[k:k+1,]
err = Hamming(x, X[k:k+1,:])
print('Class '+str(k)+' with '+str(err)+' errors')
x_orig = copy.deepcopy(x)
print(x_orig)

# Notice how the inner product of the perturbed vector is almost
# othogonal to all the other equilibria. ie. relatively close to zero
print('Inner Products: '+str(x@X.T))

## Iterate Network (to steady state)

In [ ]:
if True:
    x = copy.deepcopy(x_orig)
    n_iters = 10
    for k in range(n_iters):
        #print(x)
        x_new = Update(W, x, b)
        print(Hamming(x, x_new))
        x = x_new
else:
    x = copy.deepcopy(x_orig)
    n_iters = 40
    for count in range(n_iters):
        node_idx = list(range(N))
        np.random.shuffle(node_idx)
        for k in node_idx:
            ic = x@W[:,k] - b[k]
            x[0,k] = Thresh(ic)

## How close are we to recalling a memory?

In [ ]:
for idx,t in enumerate(X):
    d = Hamming(x, [t])[0]
    print('Memory '+str(idx)+' has error '+str(d))

# Letters Example

## Read in the letters

In [ ]:
from imageio import imread

In [ ]:
letters = []
letters.append(imread('images/A.png'))
letters.append(imread('images/B.png'))
letters.append(imread('images/C.png'))
letters.append(imread('images/E.png'))
n = len(letters)
N = len(letters[0].flatten())
X = np.zeros((n, N))
for idx,img in enumerate(letters):
    X[idx,:] = Thresh(np.array([img.flatten()-0.5]))

In [ ]:
plt.figure(figsize=(16,4))
for k in range(n):
    plt.subplot(1,n,k+1);
    plt.imshow(np.reshape(X[k], (6,6)), cmap='tab20c_r'); plt.axis('off');

In [ ]:
# Notice how the inner product of the perturbed vector is almost
# othogonal to all the other equilibria. ie. relatively close to zero
X@X.T

In [ ]:
# Connection weights and biases
b = np.zeros((1,N))
b = np.sum(X, axis=0) / n
W = ( X.T @ X ) / n - np.eye(N)
W0 = copy.deepcopy(W)

<div class="alert alert-block alert-info">
    You can try different amounts of corruption by changing <code>p</code> in the <code>Perturb</code> call below. Can you get the Hopfield network to converge to a state that it wasn't trained on?
</div>

In [ ]:
# Perturbed memory
k = np.random.randint(n)
Y = Perturb( X , p=0.2)
x = Y[k:k+1,]
x[0,24:] = -1.
err = Hamming(x, X[k:k+1,:])
print('Class '+str(k)+' with '+str(err)+' errors')
x_orig = copy.deepcopy(x)
#plt.imshow(np.reshape(x,[28,28]), cmap='gray'); plt.axis('off');
plt.imshow(np.reshape(x,[6,6]), cmap='tab20c_r'); plt.axis('off');

In [ ]:
xs = copy.deepcopy(x_orig)
xa = copy.deepcopy(x)

In [ ]:
# Synchronous updating
n_iters = 2
for idx in range(n_iters):
    xs = Update(W, xs, b)

In [ ]:
# Asynchronous updating
n_iters = 10
for count in range(n_iters):
    node_idx = list(range(N))
    np.random.shuffle(node_idx)
    for idx in node_idx:
        ic = xa@W[:,idx] - b[idx]
        xa[0,idx] = Thresh(ic)

In [ ]:
print('Correct class is '+str(k))
print('Synchronous updating')
for idx,t in enumerate(X):
    ds = Hamming(xs, [t])[0]
    print('Memory '+str(idx)+' has error '+str(ds))
print('Asynchronous updating')
for idx,t in enumerate(X):
    da = Hamming(xa, [t])[0]
    print('Memory '+str(idx)+' has error '+str(da))

In [ ]:
plt.subplot(1,2,1); plt.imshow(np.reshape(xs,[6,6]), cmap='tab20c_r'); plt.title('Synchronous'); plt.axis('off');
plt.subplot(1,2,2); plt.imshow(np.reshape(xa,[6,6]), cmap='tab20c_r'); plt.title('Asynchronous'); plt.axis('off');